[![img/cloudevel.png](img/cloudevel.png)](https://cloudevel.com)

# Apéndice: de *DuckDB* a *BigQuery*.

**sql101 · SQL para Ingeniería y Analítica de Datos** · Cloudevel® · Instructor: José Luis Chiquete · Apéndice: *De DuckDB a BigQuery*

En este *notebook* se trasladan las ideas del curso a *BigQuery*: su dialecto, el costo por bytes procesados, el particionamiento y los campos anidados.

Todo lo visto en el curso se traslada a un *data warehouse* en la nube. Este apéndice muestra cómo ejecutar las mismas ideas en [*BigQuery*](https://cloud.google.com/bigquery/docs/introduction), el *data warehouse* de *Google Cloud*, sobre el *dataset* público real ```bigquery-public-data.thelook_ecommerce```, cuyo esquema replican los datos del curso.

## Acceso sin costo.

El [*sandbox* de *BigQuery*](https://cloud.google.com/bigquery/docs/sandbox) permite usar el servicio sin tarjeta de crédito, con estas condiciones:

* 10 GB de almacenamiento y 1 TB de datos consultados al mes.
* Las tablas y particiones expiran a los 60 días.
* **No permite *DML*** (```INSERT```, ```UPDATE```, ```DELETE```, ```MERGE```). Para ellas se requiere una cuenta de facturación; el nivel gratuito sigue aplicando.

Hay dos formas de ejecutar las consultas de este apéndice:

1. **En la consola** (recomendada): abre https://console.cloud.google.com/bigquery, copia la consulta (sin la línea ```%%bigquery```) y ejecútala.
2. **En este notebook**: requiere autenticarse con la [*CLI* de *Google Cloud*](https://cloud.google.com/sdk/docs/install) mediante ```gcloud auth application-default login``` y escribir el identificador de tu proyecto en la siguiente celda.

In [ ]:
import bigquery_magics

PROYECTO = 'mi-proyecto'  # reemplaza por el identificador de tu proyecto
bigquery_magics.context.project = PROYECTO
%load_ext bigquery_magics

## Diferencias de dialecto.

*BigQuery* usa el dialecto *GoogleSQL*. Estas son las diferencias con lo aprendido en el curso:

| Tema | *ANSI* / *DuckDB* | *BigQuery* |
|---|---|---|
| Identificadores especiales | ```"mi tabla"``` | ```` `mi-proyecto.dataset.tabla` ```` |
| Cadenas de texto | ```'texto'``` | ```'texto'``` o ```"texto"``` |
| Limitar filas | ```FETCH FIRST n ROWS ONLY``` / ```LIMIT n``` | Sólo ```LIMIT n``` |
| Conteo condicional | ```COUNT(*) FILTER (WHERE ...)``` | ```COUNTIF(...)``` |
| Fecha y hora sin zona | ```TIMESTAMP``` | ```DATETIME``` |
| Instante en *UTC* | ```TIMESTAMP WITH TIME ZONE``` | ```TIMESTAMP``` |
| Diferencia entre fechas | ```EXTRACT(DAY FROM (t2 - t1))``` | ```TIMESTAMP_DIFF(t2, t1, DAY)``` |
| Truncar fechas | ```DATE_TRUNC('month', f)``` (*DuckDB*) | ```DATE_TRUNC(f, MONTH)``` |
| Unión de conjuntos | ```UNION``` (implica ```DISTINCT```) | Obligatorio ```UNION ALL``` o ```UNION DISTINCT``` |
| Texto | ```VARCHAR(n)``` | ```STRING``` |
| Restricciones | ```PRIMARY KEY```, ```CHECK``` se validan | No se validan |
| Registros anidados | Varía según el motor | ```STRUCT(1 AS a)```, ```ARRAY_AGG```, ```UNNEST``` |
| ```UPDATE``` / ```DELETE``` | ```WHERE``` opcional | ```WHERE``` obligatorio |

## La misma consulta en *BigQuery*.

Ventas por categoría (lección 4), sobre los datos públicos. Sólo cambian los nombres de las tablas:

In [ ]:
%%bigquery
SELECT p.category,
  COUNT(*) AS articulos,
  ROUND(SUM(oi.sale_price), 2) AS ventas
FROM `bigquery-public-data.thelook_ecommerce.order_items` AS oi
JOIN `bigquery-public-data.thelook_ecommerce.products` AS p
  ON oi.product_id = p.id
WHERE oi.status NOT IN ('Cancelled', 'Returned')
GROUP BY p.category
ORDER BY ventas DESC;

## Costo: bytes procesados.

En el modelo bajo demanda, *BigQuery* **cobra por los bytes que lee cada consulta**. Como almacena los datos por columnas, sólo lee las columnas que la consulta menciona. Antes de ejecutar, el editor de la consola muestra cuántos bytes se procesarán; en el notebook, la opción ```--dry_run``` hace lo mismo sin ejecutar ni cobrar.

Compara estas tres consultas sobre una tabla grande:

In [ ]:
%%bigquery --dry_run
SELECT *
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
LIMIT 10;

In [ ]:
%%bigquery --dry_run
SELECT unique_key, trip_total
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`
LIMIT 10;

In [ ]:
%%bigquery --dry_run
SELECT unique_key, trip_total
FROM `bigquery-public-data.chicago_taxi_trips.taxi_trips`;

* La primera lee **todas** las columnas de la tabla completa, aunque sólo muestre 10 filas.
* La segunda y la tercera procesan **los mismos bytes**: ```LIMIT``` no reduce la búsqueda ni el costo.
* Para explorar una tabla sin costo, usa la pestaña **Vista previa** de la consola.

## Particionamiento y *clustering*.

> **Fuera del estándar *ANSI*:** cada motor implementa el particionamiento a su manera.

Una **tabla particionada** divide sus datos en segmentos, normalmente por fecha. Cuando una consulta filtra por la columna de partición, *BigQuery* **sólo lee las particiones necesarias** (*partition pruning*). A diferencia de ```LIMIT```, esto **sí reduce los bytes procesados y el costo**. El ***clustering*** ordena los datos dentro de cada partición para reducir aún más lo que se lee al filtrar por esas columnas.

* Crea un *dataset* en la multirregión ```US```, la misma de los datos públicos (una consulta no puede leer de una ubicación y escribir en otra).

In [ ]:
%%bigquery
CREATE SCHEMA IF NOT EXISTS curso_sql OPTIONS (location = 'US');

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE curso_sql.ordenes
PARTITION BY DATE(created_at)
CLUSTER BY status
AS
SELECT order_id, user_id, status, gender, created_at, shipped_at, delivered_at, num_of_item
FROM `bigquery-public-data.thelook_ecommerce.orders`;

* Compara los bytes estimados:

In [ ]:
%%bigquery --dry_run
SELECT COUNT(*) AS ordenes
FROM curso_sql.ordenes
WHERE status = 'Complete';

In [ ]:
%%bigquery --dry_run
SELECT COUNT(*) AS ordenes
FROM curso_sql.ordenes
WHERE created_at >= TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 7 DAY)
  AND status = 'Complete';

> **Nota sobre el *sandbox*:** las particiones expiran a los 60 días **contados desde la fecha de la partición**, así que la tabla anterior sólo conservará las órdenes de los últimos 60 días. Con una cuenta de facturación se conservan todas.

https://cloud.google.com/bigquery/docs/partitioned-tables

## ```MERGE``` en *BigQuery*.

La sintaxis es la misma que en la lección 7. Requiere una cuenta de facturación.

In [ ]:
%%bigquery
CREATE OR REPLACE TABLE curso_sql.clientes AS
SELECT id, first_name, last_name, country, state, created_at
FROM `bigquery-public-data.thelook_ecommerce.users`
WHERE country = 'Brasil';

In [ ]:
%%bigquery
MERGE INTO curso_sql.clientes AS destino
USING (
  SELECT id, first_name, last_name, country, 'Actualizado' AS state, created_at
  FROM `bigquery-public-data.thelook_ecommerce.users`
  WHERE country IN ('Brasil', 'Japan') AND MOD(id, 100) = 0
) AS cambios
  ON destino.id = cambios.id
WHEN MATCHED THEN
  UPDATE SET state = cambios.state
WHEN NOT MATCHED THEN
  INSERT (id, first_name, last_name, country, state, created_at)
  VALUES (cambios.id, cambios.first_name, cambios.last_name, cambios.country, cambios.state, cambios.created_at);

## Campos anidados.

Como se mencionó en la lección 8, *BigQuery* recomienda denormalizar guardando **registros anidados**: una fila por orden, con sus artículos en un arreglo (```ARRAY```) de registros (```STRUCT```). Así no se repiten los datos de la orden en cada artículo y no se necesita ```JOIN``` al consultar.

In [ ]:
%%bigquery
SELECT o.order_id,
  o.status,
  ARRAY_AGG(STRUCT(p.name AS producto, p.category AS categoria, oi.sale_price AS precio)) AS articulos
FROM `bigquery-public-data.thelook_ecommerce.orders` AS o
JOIN `bigquery-public-data.thelook_ecommerce.order_items` AS oi ON o.order_id = oi.order_id
JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON oi.product_id = p.id
WHERE o.order_id <= 10
GROUP BY o.order_id, o.status;

* ```UNNEST``` convierte el arreglo de nuevo en filas:

In [ ]:
%%bigquery
WITH ordenes AS (
  SELECT o.order_id,
    ARRAY_AGG(STRUCT(p.category AS categoria, oi.sale_price AS precio)) AS articulos
  FROM `bigquery-public-data.thelook_ecommerce.orders` AS o
  JOIN `bigquery-public-data.thelook_ecommerce.order_items` AS oi ON o.order_id = oi.order_id
  JOIN `bigquery-public-data.thelook_ecommerce.products` AS p ON oi.product_id = p.id
  GROUP BY o.order_id
)
SELECT a.categoria, COUNT(DISTINCT o.order_id) AS ordenes, ROUND(SUM(a.precio), 2) AS ventas
FROM ordenes AS o
CROSS JOIN UNNEST(o.articulos) AS a
GROUP BY a.categoria
ORDER BY ventas DESC;

## Limpieza.

In [ ]:
%%bigquery
DROP SCHEMA IF EXISTS curso_sql CASCADE;

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>